In [ ]:
from data import *

In [ ]:
train, test, sub = get_train_test()
train_added = add_extra_data(train)
train_added.rename(columns={'SMILES':'SMILES_raw'}, inplace=True)
train_added['SMILES'] = train_added['SMILES_raw'].apply(replace_all_R_with_C)
train_cleaned = clean_smiles(train_added)
train_filtered = filter_train_data(train_cleaned)

In [ ]:
# symbols = count_symbols_in_df(train_filtered, "SMILES")

In [ ]:
# print(symbols)

In [ ]:
for z in [2, 10, 18, 1, 29, 7]:  # He, Ne, Ar
    print(z, "grid:", z_to_grid_xy(z))

In [ ]:
smi = "CC(=O)OC1=CC=CC=C1C(=O)O"  # Aspirin
mol = Chem.MolFromSmiles(smi)
for atom in mol.GetAtoms():
    print(f'name:{atom.GetSymbol()},features:{make_node_features(atom)}')

In [ ]:
# language: python
# Notebook cell: 直接验证 make_node_features / make_edge_features 输出
import math, random
from rdkit import Chem
from rdkit.Chem import rdPartialCharges
from collections import Counter
from data import make_node_features, make_edge_features

random.seed(0)
# 使用 train_filtered 优先，否则示例
try:
    smiles_all = list(train_filtered["SMILES"].astype(str))
except Exception:
    print("Using example SMILES")
    smiles_all = ["CCO","C","c1ccccc1","C/C=C/C","C#N","O=C=O","[NH4+]"]

sample = random.sample(smiles_all, min(8, len(smiles_all)))

bad_nodes = 0
bad_edges = 0
node_counts = Counter()
edge_counts = Counter()

def is_ok_num(v):
    return isinstance(v, (int,float)) and not (isinstance(v,float) and math.isnan(v))

for smi in sample:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        print("INVALID SMILES:", smi); continue
    try:
        rdPartialCharges.ComputeGasteigerCharges(mol, throwOnParamFailure=False)
    except Exception:
        pass
    print(f"\nSMILES: {smi}  | atoms: {mol.GetNumAtoms()}  bonds: {mol.GetNumBonds()}")
    for atom in mol.GetAtoms():
        try:
            feats = make_node_features(atom)
        except Exception as e:
            print("  atom", atom.GetIdx(), atom.GetSymbol(), "-> make_node_features ERROR:", e)
            bad_nodes += 1
            continue
        # basic numeric check
        non_numeric_idxs = [i for i,v in enumerate(feats) if not is_ok_num(v)]
        if non_numeric_idxs:
            print("  atom", atom.GetIdx(), atom.GetSymbol(), "-> non-numeric idxs:", non_numeric_idxs)
            bad_nodes += 1
            node_counts[tuple(non_numeric_idxs)] += 1
        # try detect hybridization one-hot: we expect a block of 1/0s
        # heuristic: find contiguous block of ints (0/1) longer than 2 near middle
        L = len(feats)
        # assume hybrid one-hot starts after first 6 numeric fields per our implementation
        hyb_start = 6
        hyb_len = None
        if L > hyb_start+2:
            # take next up to 12 entries as candidate
            cand = feats[hyb_start: min(hyb_start+12, L)]
            # treat ints 0/1 as one-hot candidates
            if all(isinstance(x,(int,float)) for x in cand):
                hyb_len = len(cand)
                hyb_sum = sum(float(x) for x in cand)
                print("  atom", atom.GetIdx(), atom.GetSymbol(), "-> feat_len", L, ", hyb_sum", hyb_sum, "hyb_len_candidate", hyb_len)
        # print short sample
        print("  idx=", atom.GetIdx(), atom.GetSymbol(), "-> feat_len", L, "sample:", feats[:min(20,L)])

    for bond in mol.GetBonds():
        try:
            e = make_edge_features(bond)
        except Exception as ex:
            print("  bond", bond.GetBeginAtomIdx(), "-", bond.GetEndAtomIdx(), "-> make_edge_features ERROR:", ex)
            bad_edges += 1
            continue
        non_numeric = [i for i,v in enumerate(e) if not is_ok_num(v)]
        if non_numeric:
            print("  bond", bond.GetBeginAtomIdx(), "-", bond.GetEndAtomIdx(), "-> non-numeric idxs:", non_numeric)
            bad_edges += 1
            edge_counts[tuple(non_numeric)] += 1
        print("  bond", bond.GetBeginAtomIdx(), "-", bond.GetEndAtomIdx(), "-> len", len(e), "sample:", e)

print("\nSUMMARY: bad_nodes:", bad_nodes, "bad_edges:", bad_edges)
print("node non-numeric groups:", dict(node_counts))
print("edge non-numeric groups:", dict(edge_counts))

In [ ]:
from tqdm import tqdm
# 使用全部 SMILES（而不是随机抽样）
sample = list(smiles_all)

# 是否打印每个原子/键的详细输出（数据多时设为 False）
verbose = True
max_examples_print = 10000  # 如果 verbose=True，最多完整打印多少个分子

seen_examples = 0

for smi in tqdm(sample, desc="checking SMILES"):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        if verbose and seen_examples < max_examples_print:
            print("INVALID SMILES:", smi)
        continue
    try:
        rdPartialCharges.ComputeGasteigerCharges(mol, throwOnParamFailure=False)
    except Exception:
        pass

    if verbose and seen_examples < max_examples_print:
        print(f"\nSMILES: {smi}  | atoms: {mol.GetNumAtoms()}  bonds: {mol.GetNumBonds()}")

    for atom in mol.GetAtoms():
        try:
            feats = make_node_features(atom)
        except Exception as e:
            if verbose and seen_examples < max_examples_print:
                print("  atom", atom.GetIdx(), atom.GetSymbol(), "-> make_node_features ERROR:", e)
            bad_nodes += 1
            continue
        non_numeric_idxs = [i for i,v in enumerate(feats) if not is_ok_num(v)]
        if non_numeric_idxs:
            if verbose and seen_examples < max_examples_print:
                print("  atom", atom.GetIdx(), atom.GetSymbol(), "-> non-numeric idxs:", non_numeric_idxs)
            bad_nodes += 1
            node_counts[tuple(non_numeric_idxs)] += 1
        if verbose and seen_examples < max_examples_print:
            # hybrid one-hot detection 和短输出保持不变
            L = len(feats)
            hyb_start = 6
            if L > hyb_start+2:
                cand = feats[hyb_start: min(hyb_start+12, L)]
                if all(isinstance(x,(int,float)) for x in cand):
                    hyb_sum = sum(float(x) for x in cand)
                    print("  atom", atom.GetIdx(), atom.GetSymbol(), "-> feat_len", L, ", hyb_sum", hyb_sum, "hyb_len_candidate", len(cand))
            print("  idx=", atom.GetIdx(), atom.GetSymbol(), "-> feat_len", L, "sample:", feats[:min(20,L)])

    for bond in mol.GetBonds():
        try:
            e = make_edge_features(bond)
        except Exception as ex:
            if verbose and seen_examples < max_examples_print:
                print("  bond", bond.GetBeginAtomIdx(), "-", bond.GetEndAtomIdx(), "-> make_edge_features ERROR:", ex)
            bad_edges += 1
            continue
        non_numeric = [i for i,v in enumerate(e) if not is_ok_num(v)]
        if non_numeric:
            if verbose and seen_examples < max_examples_print:
                print("  bond", bond.GetBeginAtomIdx(), "-", bond.GetEndAtomIdx(), "-> non-numeric idxs:", non_numeric)
            bad_edges += 1
            edge_counts[tuple(non_numeric)] += 1
        if verbose and seen_examples < max_examples_print:
            print("  bond", bond.GetBeginAtomIdx(), "-", bond.GetEndAtomIdx(), "-> len", len(e), "sample:", e)

    seen_examples += 1

In [ ]:
from data import attribute_stats_from_smiles
# 所有 SMILES 上统计原子质量（node.mass）
stats = attribute_stats_from_smiles(train_filtered["SMILES"], feature_source="node", attribute="mass", return_per_molecule=False)
print(stats)

In [ ]:
stats = attribute_stats_from_smiles(train_filtered["SMILES"], feature_source="node", attribute="is_aromatic", return_per_molecule=False)
print(stats)

In [ ]:
stats = attribute_stats_from_smiles(train_filtered["SMILES"], feature_source="node", attribute="formal_charge", return_per_molecule=False)
print(stats)

In [ ]:
stats = attribute_stats_from_smiles(train_filtered["SMILES"], feature_source="node", attribute="distance_norm", return_per_molecule=False)
print(stats)

In [ ]:
stats = attribute_stats_from_smiles(train_filtered["SMILES"], feature_source="edge", attribute="bond_stereo", return_per_molecule=False)
print(stats)

In [ ]:
# 或统计键的 is_conjugated
stats2 = attribute_stats_from_smiles(train_filtered["SMILES"], feature_source="edge", attribute="is_conjugated", return_per_molecule=True)
print(stats2["per_molecule"][:5])